# FedPerGC — Federated Personalized Learning with Gradient-based Contribution

Refactored, production-oriented version of `FedPerGC.ipynb`.

Changes made vs. the original notebook:
- Removed duplicate cells (duplicate GPU-check cell, duplicate `train_one_round_with_history_with_contribution` definition, duplicate history-init blocks, empty/no-op cells).
- Removed near-duplicate evaluation cells (3+ copies of the same ~150 line block that only changed a `round` number or a `test_dir`) and replaced them with a single reusable `evaluate_model_on_dir` function plus thin wrappers (`evaluate_round_range`, `evaluate_per_client`).
- Introduced a `FLConfig` dataclass instead of scattered global variables (`pass_name`, `batch_size`, `rounds`, ...).
- Split the ~450 line monolithic training function into small, testable, single-purpose functions (client sync, local training, aggregation, evaluation, history persistence).
- Replaced module-level mutable history lists with explicit function return values.
- Added type hints and docstrings to every public function.

## Table of contents
1. Environment setup
2. Configuration
3. Model architecture
4. Data pipeline
5. Training utilities (callbacks, class weights)
6. System resource monitoring
7. Federated-learning analytics utilities
8. Federated round core logic
9. Federated learning orchestration
10. History persistence
11. Model loading & evaluation
12. Benchmarking
13. Visualization
14. Main execution

## 1. Environment setup

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf


def check_gpu() -> None:
    """Verify that a GPU is visible to TensorFlow; raise if none is found."""
    device_name = tf.test.gpu_device_name()
    if device_name != "/device:GPU:0":
        raise SystemError("GPU device not found")
    print(f"Found GPU at: {device_name}")


def mount_drive() -> None:
    """Mount Google Drive when running on Colab; no-op otherwise."""
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except ImportError:
        print("Not running on Google Colab; skipping drive mount.")


check_gpu()
mount_drive()

## 2. Configuration

All parameters that were previously scattered as loose globals are centralized here.

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Dict, List


@dataclass
class FLConfig:
    """Central configuration for the FedPerGC experiment."""

    pass_name: str = "FL_April_13th"
    img_size: int = 224
    batch_size: int = 32
    epochs_local: int = 2
    rounds: int = 23
    l2_lambda: float = 0.001
    base_dir: str = "/content/drive/MyDrive/FL"
    all_classes: List[str] = field(
        default_factory=lambda: ["Brown_Spot", "Leaf_Blast", "Leaf_Blight", "Normal"]
    )

    @property
    def global_data_path(self) -> str:
        return f"{self.base_dir}/global"

    @property
    def local_model_dir(self) -> str:
        return f"{self.base_dir}/local_models/{self.pass_name}"

    @property
    def model_dir(self) -> str:
        return f"{self.base_dir}/model/{self.pass_name}"

    @property
    def history_dir(self) -> str:
        return f"{self.base_dir}/history/{self.pass_name}"

    @property
    def contribution_dir(self) -> str:
        return f"{self.base_dir}/contribution/{self.pass_name}"

    @property
    def gradient_similarity_dir(self) -> str:
        return f"{self.base_dir}/gradient_similarity/{self.pass_name}"

    @property
    def resource_history_dir(self) -> str:
        return f"{self.base_dir}/resource_history/{self.pass_name}"


config = FLConfig()

CLIENTS_INFO: Dict[str, dict] = {
    f"Client_{i}": {
        "path": f"/content/drive/MyDrive/FL/dataclients/six_clients/apha08/Client_{i}",
        "classes": config.all_classes,
    }
    for i in range(1, 7)
}

## 3. Model architecture

CBAM-style attention residual backbone (`build_efficient_fl_network`). Single definition, registered as Keras-serializable so it can be reloaded from `.keras` checkpoints.

In [ ]:
import math

from tensorflow.keras import Model, layers


@tf.keras.utils.register_keras_serializable()
class GlobalAveragePooling2DLayer(layers.Layer):
    """Channel-wise average pooling that keeps a singleton channel dimension."""

    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)


@tf.keras.utils.register_keras_serializable()
class GlobalMaxPooling2DLayer(layers.Layer):
    """Channel-wise max pooling that keeps a singleton channel dimension."""

    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)


@tf.keras.utils.register_keras_serializable()
def improve_cbam_block(input_feature, gamma: int = 2, b: int = 1):
    """Lightweight hybrid channel + spatial attention block (improved CBAM)."""
    channel = input_feature.shape[-1]
    t = int(abs((math.log2(channel) + b) / gamma))
    k = max(3, t if t % 2 else t + 1)

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    channel_attention = layers.DepthwiseConv2D(
        kernel_size=(k, 1),
        padding="same",
        activation="sigmoid",
        use_bias=False,
        depthwise_initializer="he_normal",
    )(avg_pool)
    channel_refined = layers.Multiply()([input_feature, channel_attention])

    avg_spatial = GlobalAveragePooling2DLayer()(channel_refined)
    max_spatial = GlobalMaxPooling2DLayer()(channel_refined)
    concat_spatial = layers.Concatenate(axis=-1)([avg_spatial, max_spatial])
    spatial_attention = layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        kernel_initializer="he_normal",
    )(concat_spatial)

    return layers.Multiply()([channel_refined, spatial_attention])


@tf.keras.utils.register_keras_serializable()
def residual_improve_cbam_block(x, filters: int, strides: int = 1, dropout_rate: float = 0.1, use_attention: bool = True):
    """Residual block with a separable convolution main path and optional improved-CBAM attention."""
    shortcut = x

    x = layers.SeparableConv2D(
        filters, kernel_size=3, strides=strides, padding="same",
        depthwise_initializer="he_normal", pointwise_initializer="he_normal",
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(max_value=6)(x)

    if use_attention:
        x = improve_cbam_block(x)

    if shortcut.shape[-1] != filters or strides != 1:
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding="same", kernel_initializer="he_normal")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU(max_value=6)(x)
    x = layers.Dropout(dropout_rate)(x)
    return x


@tf.keras.utils.register_keras_serializable()
def build_efficient_fl_network(
    input_shape=(224, 224, 3),
    num_classes: int = 4,
    dropout_rate: float = 0.2,
    l2_reg: float = 1e-5,
) -> Model:
    """Build the CBAM-residual backbone used for both the global and client models."""
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(16, (3, 3), strides=2, padding="same", activation="swish",
                       kernel_initializer="he_normal", kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), strides=1, padding="same", activation="swish",
                       kernel_initializer="he_normal", kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)

    # (filters, strides, use_attention, num_blocks) per backbone stage
    backbone_config = [
        (32, 1, True, 2),
        (64, 2, True, 2),
        (128, 2, True, 2),
        (256, 2, True, 1),
        (512, 2, True, 1),
        (1024, 2, False, 1),
    ]

    for stage_idx, (filters, strides, use_attn, num_blocks) in enumerate(backbone_config):
        stage_dropout = dropout_rate * (0.75 ** stage_idx)
        for block_idx in range(num_blocks):
            block_strides = strides if block_idx == 0 else 1
            x = residual_improve_cbam_block(
                x, filters, strides=block_strides,
                dropout_rate=max(0.05, stage_dropout), use_attention=use_attn,
            )

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(96, activation="swish", kernel_initializer="he_normal",
                      kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="swish", kernel_initializer="he_normal")(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation="softmax", kernel_initializer="he_normal")(x)

    return Model(inputs, outputs, name="proposal_3")


def model_fn(input_shape, num_classes: int, l2_reg_value: float) -> Model:
    """Factory used by the FL training loop to (re)build a fresh client/global model."""
    return build_efficient_fl_network(input_shape=input_shape, num_classes=num_classes, l2_reg=l2_reg_value)


def get_custom_objects() -> dict:
    """Custom objects required to deserialize saved `.keras` checkpoints."""
    return {
        "GlobalAveragePooling2DLayer": GlobalAveragePooling2DLayer,
        "GlobalMaxPooling2DLayer": GlobalMaxPooling2DLayer,
        "improve_cbam_block": improve_cbam_block,
        "residual_improve_cbam_block": residual_improve_cbam_block,
        "build_efficient_fl_network": build_efficient_fl_network,
    }

## 4. Data pipeline

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


def count_images_by_class(data_dir: str, classes: List[str]) -> Dict[str, int]:
    """Count image files per class folder under `data_dir`."""
    counts = {}
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        if os.path.exists(class_dir):
            counts[class_name] = sum(
                1 for entry in os.scandir(class_dir)
                if entry.is_file() and entry.name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif"))
            )
        else:
            counts[class_name] = 0
    return counts


def _fix_duplicate_path_segment(folder_path: str) -> str:
    parts = os.path.normpath(folder_path).split(os.sep)
    if len(parts) >= 2 and parts[-1] == parts[-2]:
        fixed_path = os.path.join(*parts[:-1])
        print(f"[INFO] Fixed duplicate path: {folder_path} -> {fixed_path}")
        return fixed_path
    return folder_path


def load_data(
    train_folder: str,
    classes=None,
    valid_folder: str = None,
    img_size=(224, 224),
    batch_size: int = 32,
):
    """Build train/validation `ImageDataGenerator` flows for one client or the global dataset."""
    train_folder = _fix_duplicate_path_segment(train_folder)
    if valid_folder:
        valid_folder = _fix_duplicate_path_segment(valid_folder)

    if not os.path.isdir(train_folder):
        print(f"[ERROR] Training folder not found: {train_folder}")
        return None, None

    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255, rotation_range=25, width_shift_range=0.15,
        height_shift_range=0.15, shear_range=0.1, zoom_range=0.2,
    )
    val_datagen = ImageDataGenerator(rescale=1.0 / 255)

    try:
        train_ds = train_datagen.flow_from_directory(
            directory=train_folder, target_size=img_size, batch_size=batch_size,
            class_mode="categorical", classes=classes, shuffle=True,
        )
        print(f"[INFO] Loaded training data: {train_ds.samples} images, {train_ds.num_classes} classes.")
    except Exception as exc:
        print(f"[ERROR] Failed to load training data: {exc}")
        return None, None

    val_ds = None
    if valid_folder and os.path.isdir(valid_folder):
        try:
            val_ds = val_datagen.flow_from_directory(
                directory=valid_folder, target_size=img_size, batch_size=batch_size,
                class_mode="categorical", classes=classes, shuffle=False,
            )
            print(f"[INFO] Loaded validation data: {val_ds.samples} images, {val_ds.num_classes} classes.")
        except Exception as exc:
            print(f"[ERROR] Failed to load validation data: {exc}")
    elif valid_folder:
        print(f"[WARNING] Validation folder not found: {valid_folder}")

    return train_ds, val_ds

## 5. Training utilities (callbacks, class weights, LR schedule)

In [ ]:
import time

import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import Callback


class TimeHistory(Callback):
    """Records wall-clock duration of every training epoch."""

    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.time() - self._epoch_start)


def class_weights_func(train_dir: str) -> Dict[int, float]:
    """Compute balanced class weights from the class-folder counts under `train_dir`."""
    classes = sorted(os.listdir(train_dir))
    class_indices = {cls: i for i, cls in enumerate(classes)}
    labels = []
    for cls in classes:
        count = len(os.listdir(os.path.join(train_dir, cls)))
        labels.extend([class_indices[cls]] * count)

    weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
    weights_dict = dict(enumerate(weights))
    print("Class Weights:", weights_dict)
    return weights_dict


def lr_scheduler(epoch: int, lr: float) -> float:
    """Halve the learning rate for the first 3 warm-up epochs."""
    return lr * 0.5 if epoch < 3 else lr


lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_scheduler)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)

## 6. System resource monitoring

In [ ]:
import csv
from datetime import datetime
from typing import Optional, Tuple

try:
    import psutil
except ImportError:
    psutil = None

try:
    import GPUtil
except ImportError:
    GPUtil = None


def get_gpu_usage() -> Tuple[Optional[float], Optional[float]]:
    """Return (usage_percent, used_mb) for the first visible GPU, or (None, None)."""
    if GPUtil is None:
        return None, None
    try:
        gpus = GPUtil.getGPUs()
        if gpus:
            gpu = gpus[0]
            return gpu.memoryUtil * 100, gpu.memoryUsed
    except Exception:
        pass
    return None, None


def log_system_usage(round_idx: int, csv_path: str) -> None:
    """Append CPU/RAM/GPU/disk usage for the current round to a CSV log."""
    if psutil is None:
        print("[WARN] psutil not installed; skipping system usage log.")
        return

    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    cpu_usage = psutil.cpu_percent(interval=1)
    ram = psutil.virtual_memory()
    disk = psutil.disk_usage("/")
    gpu_usage_percent, gpu_memory = get_gpu_usage()

    row = [
        round_idx,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        cpu_usage,
        ram.percent,
        ram.used / (1024 ** 3),
        gpu_usage_percent if gpu_usage_percent is not None else "N/A",
        gpu_memory if gpu_memory is not None else "N/A",
        disk.percent,
        disk.used / (1024 ** 3),
    ]

    write_header = not os.path.exists(csv_path)
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow([
                "Round", "Timestamp", "CPU_Usage(%)", "RAM_Usage(%)", "RAM_System",
                "GPU_Usage(%)", "RAM_GPU", "Disk_Usage(%)", "Disk_Space",
            ])
        writer.writerow(row)

## 7. Federated-learning analytics utilities

JSON persistence helpers, communication-overhead estimation, client-contribution scoring, weight padding, and gradient cosine similarity.

In [ ]:
import json

from sklearn.metrics.pairwise import cosine_similarity


def default_converter(o):
    """`json.dump` default= hook that converts numpy scalars/arrays to native types."""
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Type {type(o)} not serializable")


def save_json(path: str, data) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, default=default_converter, indent=4)


def append_json_history(path: str, entry: dict) -> None:
    """Load the JSON list at `path` (or start a new one), append `entry`, and save it back."""
    history = []
    if os.path.exists(path):
        with open(path, "r") as f:
            history = json.load(f)
    history.append(entry)
    save_json(path, history)


def calculate_communication_overhead(model_size_mb: float, num_clients: int, num_rounds: int, direction: str = "both") -> float:
    """Estimate total FL communication volume (MB) for upload/download/both directions."""
    if direction not in ("upload", "download", "both"):
        raise ValueError("Direction must be 'upload', 'download', or 'both'.")
    multiplier = 2 if direction == "both" else 1
    return model_size_mb * num_clients * multiplier * num_rounds


def calculate_client_contribution(global_weights_before, client_weights_after, _client_weights_after_dup, client_sample_count: int) -> float:
    """Estimate a client's contribution as its weight-update magnitude, scaled by sample count."""
    if not global_weights_before or not client_weights_after:
        return 0.0

    update_magnitude = 0.0
    for w_before, w_after in zip(global_weights_before, client_weights_after):
        if w_before.shape == w_after.shape:
            update_magnitude += np.sum(np.square(w_after - w_before))

    return update_magnitude * client_sample_count


def pad_to_shape(array: np.ndarray, shape) -> np.ndarray:
    """Zero-pad `array` up to `shape` (used to reconcile mismatched final-layer sizes)."""
    if tuple(array.shape) == tuple(shape):
        return array
    padding = [(0, shape[i] - array.shape[i]) for i in range(array.ndim)]
    return np.pad(array, padding, mode="constant")


def compute_delta_weights(before, after):
    """Element-wise `after - before` per layer, padding to reconcile shape mismatches."""
    deltas = []
    for w_before, w_after in zip(before, after):
        if w_before.shape != w_after.shape:
            target_shape = np.maximum(w_before.shape, w_after.shape)
            w_before = pad_to_shape(w_before, target_shape)
            w_after = pad_to_shape(w_after, target_shape)
        deltas.append(w_after - w_before)
    return deltas


def compute_gradient_cosine_similarities(client_gradients: Dict[str, list]) -> Dict[str, float]:
    """Mean cosine similarity of per-layer gradients between every pair of clients."""
    similarities = {}
    client_ids = list(client_gradients.keys())
    for i in range(len(client_ids)):
        for j in range(i + 1, len(client_ids)):
            id_a, id_b = client_ids[i], client_ids[j]
            scores = []
            for layer_a, layer_b in zip(client_gradients[id_a], client_gradients[id_b]):
                flat_a, flat_b = layer_a.flatten(), layer_b.flatten()
                if (
                    flat_a.shape == flat_b.shape
                    and np.linalg.norm(flat_a) > 1e-9
                    and np.linalg.norm(flat_b) > 1e-9
                ):
                    scores.append(cosine_similarity(flat_a.reshape(1, -1), flat_b.reshape(1, -1))[0][0])
            similarities[f"{id_a}_vs_{id_b}"] = float(np.mean(scores)) if scores else 0.0
    return similarities

## 8. Federated round core logic

This replaces the ~450-line monolithic `train_one_round_with_history_with_contribution` (which was also duplicated verbatim in the original notebook) with small, composable steps.

In [ ]:
def _sync_client_weights(global_model: Model, client_model: Model) -> None:
    """Copy compatible layer weights from the global model into a client model (all but final layer)."""
    for global_layer, client_layer in zip(global_model.layers[:-1], client_model.layers[:-1]):
        g_w, c_w = global_layer.get_weights(), client_layer.get_weights()
        if g_w and c_w and g_w[0].shape == c_w[0].shape:
            try:
                client_layer.set_weights(g_w)
            except Exception as exc:
                print(f"[WARN] Could not sync layer {global_layer.name}: {exc}")


def _load_or_create_client_model(client_id: str, num_classes: int, global_model: Model, model_fn, config: FLConfig):
    """Load a cached client model if present, otherwise build a fresh one, then sync weights with the global model."""
    model_path = os.path.join(config.local_model_dir, f"{client_id}_model.keras")
    client_model = None
    if os.path.exists(model_path):
        try:
            client_model = tf.keras.models.load_model(model_path, custom_objects=get_custom_objects())
        except Exception as exc:
            print(f"[WARN] Failed to load cached model for {client_id}: {exc}")

    if client_model is None:
        client_model = model_fn(input_shape=(config.img_size, config.img_size, 3), num_classes=num_classes, l2_reg_value=config.l2_lambda)
        client_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

    _sync_client_weights(global_model, client_model)
    return client_model, model_path


def train_client_locally(
    client_id: str,
    info: dict,
    global_model: Model,
    model_fn,
    config: FLConfig,
    local_epochs: int,
    round_idx: int,
    resource_dir: str,
) -> Optional[dict]:
    """Train one client's local model for `local_epochs` and return its update payload, or None if no data."""
    train_dir = os.path.join(info["path"], "train")
    classes = info["classes"]

    client_model, model_path = _load_or_create_client_model(client_id, len(classes), global_model, model_fn, config)

    train_data, _ = load_data(train_dir, classes=classes, img_size=(config.img_size, config.img_size), batch_size=config.batch_size)
    if not train_data or train_data.samples == 0:
        print(f"[WARN] No training data for {client_id}; skipping.")
        return None

    initial_weights = client_model.get_weights()
    class_weights = class_weights_func(train_dir)

    csv_path = os.path.join(resource_dir, f"{client_id}_system_usage.csv")
    log_system_usage(round_idx, csv_path)

    timer = TimeHistory()
    start = time.time()
    history = client_model.fit(train_data, epochs=local_epochs, class_weight=class_weights, verbose=1, callbacks=[timer])
    print(f"[INFO] {client_id} local training took {time.time() - start:.2f}s ({timer.times})")

    log_system_usage(round_idx, csv_path)

    try:
        client_model.save(model_path)
    except Exception as exc:
        print(f"[WARN] Failed to save local model for {client_id}: {exc}")

    final_weights = client_model.get_weights()
    deltas = compute_delta_weights(initial_weights, final_weights)

    return {
        "client_id": client_id,
        "sample_count": train_data.samples,
        "delta_weights": deltas,
        "fedper_weights": final_weights[:-2],
        "final_weights": final_weights,
        "gradients": deltas,
        "loss_history": history.history.get("loss", []),
        "acc_history": history.history.get("accuracy", []),
    }


def aggregate_client_updates(global_model: Model, updates: List[dict], weight_map: Dict[str, float], use_delta: bool = True) -> Model:
    """Aggregate client updates (delta-weighted FedAvg, or FedPer full-weights excluding the head) into the global model."""
    global_weights_before = global_model.get_weights()
    key = "delta_weights" if use_delta else "fedper_weights"
    num_layers = len(updates[0][key])
    aggregated = []

    for layer_idx in range(num_layers):
        layer_updates = [u[key][layer_idx] for u in updates]
        target_shape = np.max([lu.shape for lu in layer_updates], axis=0)
        weighted_sum = sum(
            pad_to_shape(lu, target_shape) * weight_map.get(u["client_id"], 0.0)
            for lu, u in zip(layer_updates, updates)
        )

        base_weight = global_weights_before[layer_idx]
        if base_weight.shape != weighted_sum.shape:
            merged_shape = np.maximum(base_weight.shape, weighted_sum.shape)
            base_weight = pad_to_shape(base_weight, merged_shape)
            weighted_sum = pad_to_shape(weighted_sum, merged_shape)
        aggregated.append(base_weight + weighted_sum)

    if not use_delta:
        aggregated.extend(global_model.get_weights()[-2:])

    try:
        global_model.set_weights(aggregated)
    except ValueError as exc:
        print(f"[WARN] Could not set aggregated weights directly: {exc}")
    return global_model


def evaluate_global_model(global_model: Model, config: FLConfig) -> Tuple[float, float]:
    """Evaluate the aggregated global model on the shared validation set."""
    test_dir = os.path.join(config.global_data_path, "valid")
    datagen = ImageDataGenerator(rescale=1.0 / 255)
    generator = datagen.flow_from_directory(
        test_dir, target_size=(config.img_size, config.img_size), batch_size=config.batch_size,
        class_mode="categorical", classes=config.all_classes, shuffle=False,
    )
    return global_model.evaluate(generator, verbose=0)


def train_one_round(
    global_model: Model,
    clients_info: Dict[str, dict],
    round_idx: int,
    local_epochs: int,
    model_fn,
    config: FLConfig,
    use_delta: bool = True,
):
    """Run one full FL round: local training, contribution scoring, aggregation, and global evaluation."""
    print(f"\n--- Starting Round {round_idx} ---")
    global_weights_before = global_model.get_weights()
    resource_dir = os.path.join(config.resource_history_dir, "clients")
    os.makedirs(config.local_model_dir, exist_ok=True)
    os.makedirs(resource_dir, exist_ok=True)

    updates = []
    for client_id, info in clients_info.items():
        print(f"\n======> Training on Client: {client_id}")
        result = train_client_locally(client_id, info, global_model, model_fn, config, local_epochs, round_idx, resource_dir)
        if result:
            updates.append(result)

    if not updates:
        print("[WARN] No client produced updates this round.")
        return global_model, {}, {}, {"train_acc": 0.0, "train_loss": 0.0, "val_acc": 0.0, "val_loss": 0.0}

    total_samples = sum(u["sample_count"] for u in updates)
    weighted_acc = sum((u["acc_history"][-1] if u["acc_history"] else 0.0) * u["sample_count"] for u in updates) / total_samples
    weighted_loss = sum((u["loss_history"][-1] if u["loss_history"] else 0.0) * u["sample_count"] for u in updates) / total_samples
    print(f"[INFO] Round {round_idx} weighted client accuracy: {weighted_acc:.4f}")

    append_json_history(
        os.path.join(config.history_dir, "weighted_avg_acc_history.json"),
        {"round": round_idx, "accuracy": weighted_acc},
    )

    gradient_similarities = compute_gradient_cosine_similarities({u["client_id"]: u["gradients"] for u in updates})
    print("Gradient Similarities:", gradient_similarities)
    save_json(os.path.join(config.gradient_similarity_dir, f"gradient_similarity_round_{round_idx}.json"), gradient_similarities)

    contribution_scores = {
        u["client_id"]: calculate_client_contribution(global_weights_before, u["final_weights"], u["final_weights"], u["sample_count"])
        for u in updates
    }
    total_contribution = sum(contribution_scores.values())
    if total_contribution > 0:
        weight_map = {cid: score / total_contribution for cid, score in contribution_scores.items()}
    else:
        print("[WARN] Total contribution is zero; falling back to sample-count weighting.")
        weight_map = {u["client_id"]: u["sample_count"] / total_samples for u in updates}
    save_json(os.path.join(config.contribution_dir, f"contribution_round_{round_idx}.json"), contribution_scores)

    global_model = aggregate_client_updates(global_model, updates, weight_map, use_delta=use_delta)

    global_loss, global_acc = evaluate_global_model(global_model, config)
    print(f"[INFO] Round {round_idx} global test loss={global_loss:.4f}, acc={global_acc:.4f}")

    log_system_usage(round_idx, os.path.join(config.resource_history_dir, "system_usage.csv"))

    client_histories = {u["client_id"]: {"loss": u["loss_history"], "accuracy": u["acc_history"]} for u in updates}
    round_metrics = {"train_acc": weighted_acc, "train_loss": weighted_loss, "val_acc": global_acc, "val_loss": global_loss}

    print(f"--- Finished Round {round_idx} ---")
    return global_model, contribution_scores, client_histories, round_metrics

## 9. Federated learning orchestration

In [ ]:
def run_federated_learning(global_model: Model, clients_info: Dict[str, dict], model_fn, config: FLConfig, use_delta: bool = True):
    """Run the full FL loop for `config.rounds` rounds, saving a checkpoint after every round.

    Returns (final_global_model, global_history, client_history).
    """
    global_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    client_history = {"train_loss": {}, "train_acc": {}}

    start = time.time()
    for round_idx in range(1, config.rounds + 1):
        global_model, _contribution_scores, client_histories, metrics = train_one_round(
            global_model, clients_info, round_idx, config.epochs_local, model_fn, config, use_delta=use_delta,
        )

        for key in ("train_loss", "train_acc", "val_loss", "val_acc"):
            global_history[key].append(metrics[key])

        for client_id, hist in client_histories.items():
            client_history["train_loss"].setdefault(client_id, []).extend(hist["loss"])
            client_history["train_acc"].setdefault(client_id, []).extend(hist["accuracy"])

        round_dir = os.path.join(config.model_dir, f"round_{round_idx}")
        os.makedirs(round_dir, exist_ok=True)
        global_model.save(os.path.join(round_dir, f"global_model_round_{round_idx}.keras"))

    print(f"[INFO] Federated learning finished in {time.time() - start:.2f}s")
    return global_model, global_history, client_history

## 10. History persistence

In [ ]:
def save_history(config: FLConfig, global_history: dict, client_history: dict) -> None:
    save_json(os.path.join(config.history_dir, "global_history.json"), global_history)
    save_json(os.path.join(config.history_dir, "client_history.json"), client_history)
    print(f"Global/client history saved to: {config.history_dir}")


def load_history(config: FLConfig):
    with open(os.path.join(config.history_dir, "global_history.json")) as f:
        global_history = json.load(f)
    with open(os.path.join(config.history_dir, "client_history.json")) as f:
        client_history = json.load(f)
    return global_history, client_history

## 11. Model loading & evaluation

One reusable evaluator replaces the three near-identical ~150 line evaluation cells from the original notebook (they only differed by a hard-coded `round` number or `test_dir`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import auc, classification_report, confusion_matrix, roc_curve
from sklearn.preprocessing import label_binarize


def load_fl_model(model_path: str) -> Model:
    """Load and compile a saved global/client `.keras` checkpoint."""
    model = tf.keras.models.load_model(model_path, custom_objects=get_custom_objects())
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def evaluate_model_on_dir(model: Model, test_dir: str, classes: List[str], img_size: int = 224, batch_size: int = 32, title: str = "Model") -> pd.DataFrame:
    """Evaluate `model` on a directory of test images: metrics, confusion matrix, ROC curves, and inference timing."""
    datagen = ImageDataGenerator(rescale=1.0 / 255)
    generator = datagen.flow_from_directory(
        test_dir, target_size=(img_size, img_size), batch_size=batch_size,
        class_mode="categorical", classes=classes, shuffle=False,
    )

    loss, accuracy = model.evaluate(generator, verbose=0)
    print(f"[{title}] Test Loss: {loss * 100:.2f}, Test Accuracy: {accuracy * 100:.2f}")

    y_pred = model.predict(generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = generator.classes
    num_classes = len(classes)

    report_df = pd.DataFrame(classification_report(y_true, y_pred_classes, target_names=classes, output_dict=True)).transpose()
    report_df[["precision", "recall", "f1-score"]] *= 100
    print("\nClassification Report (in %):")
    print(report_df.round(2))

    cm = confusion_matrix(y_true, y_pred_classes)
    plt.figure(figsize=(5, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {title}")
    plt.show()

    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    plt.figure(figsize=(5, 5))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
        plt.plot(fpr, tpr, label=f"{classes[i]} (AUC={auc(fpr, tpr):.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC - {title}")
    plt.legend(loc="lower right")
    plt.show()

    num_images = generator.samples
    if num_images:
        start = time.time()
        model.predict(generator, verbose=0)
        elapsed = time.time() - start
        print(f"Inference time: {elapsed:.2f}s total, {elapsed / num_images * 1000:.2f} ms/image ({num_images} images)")
    else:
        print("No test images found; skipping inference timing.")

    return report_df.round(2)


def evaluate_round_range(config: FLConfig, rounds, test_dir: str, classes: List[str], title: str = "FedPerGC") -> Dict[int, pd.DataFrame]:
    """Evaluate the saved global model checkpoint for each round index in `rounds`."""
    reports = {}
    for round_idx in rounds:
        model_path = os.path.join(config.model_dir, f"round_{round_idx}", f"global_model_round_{round_idx}.keras")
        if not os.path.exists(model_path):
            print(f"[WARN] Missing model for round {round_idx}: {model_path}")
            continue
        model = load_fl_model(model_path)
        reports[round_idx] = evaluate_model_on_dir(model, test_dir, classes, config.img_size, config.batch_size, f"{title} - round {round_idx}")
    return reports


def evaluate_per_client(fl_model: Model, clients_info: Dict[str, dict], config: FLConfig, title: str = "FedPerGC") -> Dict[str, pd.DataFrame]:
    """Evaluate the (global) model on each client's own held-out test set."""
    reports = {}
    for client_id, info in clients_info.items():
        test_dir = os.path.join(info["path"], "test")
        counts = count_images_by_class(test_dir, info["classes"])
        print(f"Client: {client_id} | Test counts: {counts} | Total: {sum(counts.values())}")
        if sum(counts.values()) == 0:
            continue
        reports[client_id] = evaluate_model_on_dir(fl_model, test_dir, info["classes"], config.img_size, config.batch_size, f"{title} | {client_id}")
    return reports

## 12. Benchmarking

In [ ]:
from tensorflow.keras.preprocessing.image import DirectoryIterator


def benchmark_model_with_format(model: Model, test_data, batch_sizes=(16, 32, 64), runs: int = 5, warmup_batches: int = 1) -> pd.DataFrame:
    """Benchmark inference FPS / latency / CPU RAM / GPU memory, reported as Mean +/- Std per batch size."""
    gpu_available = tf.config.list_physical_devices("GPU")
    results = []

    for batch_size in batch_sizes:
        fps_list, latency_list, cpu_ram_list, gpu_mem_peak_list = [], [], [], []

        for _run in range(runs):
            total_images = 0
            total_time = 0.0

            if isinstance(test_data, DirectoryIterator):
                test_data.reset()
                data_iter = ((test_data[i][0], test_data[i][1]) for i in range(len(test_data)))
            elif isinstance(test_data, tf.data.Dataset):
                data_iter = test_data.batch(batch_size)
            else:
                raise ValueError("test_data must be a DirectoryIterator or tf.data.Dataset")

            for i, (x_batch, _y_batch) in enumerate(data_iter):
                if i < warmup_batches:
                    model.predict(x_batch, verbose=0)
                    continue

                cpu_ram_before = psutil.virtual_memory().used / (1024 ** 2) if psutil else 0
                gpu_mem_before = tf.config.experimental.get_memory_info("GPU:0")["peak"] if gpu_available else 0

                start = time.time()
                model.predict(x_batch, verbose=0)
                elapsed = time.time() - start

                if psutil:
                    cpu_ram_after = psutil.virtual_memory().used / (1024 ** 2)
                    cpu_ram_list.append(cpu_ram_after - cpu_ram_before)
                if gpu_available:
                    gpu_mem_after = tf.config.experimental.get_memory_info("GPU:0")["peak"]
                    gpu_mem_peak_list.append(gpu_mem_after - gpu_mem_before)

                total_time += elapsed
                total_images += x_batch.shape[0]

            fps_list.append(total_images / total_time if total_time > 0 else 0)
            latency_list.append((total_time / total_images) * 1000 if total_images > 0 else 0)

        def format_mean_std(values):
            return f"{np.mean(values):.3f} +/- {np.std(values):.3f}" if values else "0 +/- 0"

        results.append({
            "Batch Size": batch_size,
            "FPS": format_mean_std(fps_list),
            "Latency (ms)": format_mean_std(latency_list),
            "CPU RAM (MB)": format_mean_std(cpu_ram_list),
            "GPU Memory Peak (MB)": format_mean_std([x / (1024 ** 2) for x in gpu_mem_peak_list]),
        })

    return pd.DataFrame(results)

## 13. Visualization

In [ ]:
from scipy.ndimage import gaussian_filter1d


def plot_training_curves(global_history: dict, client_history: dict, sigma: float = 1) -> None:
    """Plot smoothed global accuracy/loss curves and per-client accuracy/loss curves."""
    smoothed = {k: (gaussian_filter1d(v, sigma=sigma) if v else v) for k, v in global_history.items()}

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(smoothed["train_acc"], label="Train Acc")
    plt.plot(smoothed["val_acc"], label="Val Acc")
    plt.title("Global Accuracy over Rounds")
    plt.xlabel("Round")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(smoothed["train_loss"], label="Train Loss")
    plt.plot(smoothed["val_loss"], label="Val Loss")
    plt.title("Global Loss over Rounds")
    plt.xlabel("Round")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    for client, acc in client_history["train_acc"].items():
        plt.plot(gaussian_filter1d(acc, sigma=sigma) if acc else acc, label=client)
    plt.title("Client Training Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    for client, loss in client_history["train_loss"].items():
        plt.plot(gaussian_filter1d(loss, sigma=sigma) if loss else loss, label=client)
    plt.title("Client Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_contribution_scores(config: FLConfig) -> None:
    """Plot per-round and cumulative client contribution scores."""
    files = sorted(f for f in os.listdir(config.contribution_dir) if f.startswith("contribution_round_"))
    data = {}
    for fname in files:
        round_num = int(fname.replace("contribution_round_", "").replace(".json", ""))
        with open(os.path.join(config.contribution_dir, fname)) as f:
            data[round_num] = json.load(f)

    if not data:
        print(f"No contribution data found in {config.contribution_dir}")
        return

    df = pd.DataFrame.from_dict(data, orient="index").sort_index()
    df.index.name = "Round"

    plt.figure(figsize=(12, 6))
    for client in df.columns:
        plt.plot(df.index, df[client], marker="o", label=client)
    plt.title("Client Contribution Scores Per Round")
    plt.xlabel("Round")
    plt.ylabel("Contribution Score")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(12, 6))
    df.cumsum(axis=0).plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title("Cumulative Client Contribution Over Rounds")
    plt.xlabel("Round")
    plt.ylabel("Cumulative Contribution Score")
    plt.xticks(rotation=0)
    plt.legend(title="Client")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()


def plot_gradient_similarity(config: FLConfig) -> None:
    """Scatter-plot the pairwise gradient cosine similarity across rounds."""
    files = [f for f in os.listdir(config.gradient_similarity_dir) if f.endswith(".json")]
    data = {}
    for fname in files:
        try:
            round_num = int(fname.split("_")[-1].split(".")[0])
        except (ValueError, IndexError):
            continue
        with open(os.path.join(config.gradient_similarity_dir, fname)) as f:
            data[round_num] = json.load(f)

    if not data:
        print(f"No gradient similarity data found in {config.gradient_similarity_dir}")
        return

    rounds = sorted(data.keys())
    pairs = list(data[rounds[0]].keys())

    plt.figure(figsize=(12, 6))
    for pair in pairs:
        plt.scatter(rounds, [data[r][pair] for r in rounds], label=pair)
    plt.xlabel("Round")
    plt.ylabel("Gradient Similarity")
    plt.title("Gradient Similarity Over Rounds")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_resource_usage(csv_path: str) -> None:
    """Plot CPU/RAM/GPU usage percentages logged during training."""
    if not os.path.exists(csv_path):
        print(f"Resource usage history not found at {csv_path}.")
        return

    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 6))
    plt.plot(df["Round"], df["RAM_Usage(%)"], marker="o", linestyle="-", label="RAM Usage (%)")
    if "CPU_Usage(%)" in df.columns:
        plt.plot(df["Round"], df["CPU_Usage(%)"], marker="x", linestyle="--", label="CPU Usage (%)")
    if "GPU_Usage(%)" in df.columns and not df["GPU_Usage(%)"].dropna().empty:
        plt.plot(df["Round"], df["GPU_Usage(%)"], marker="s", linestyle=":", label="GPU Usage (%)")

    plt.title("Resource Usage Over Federated Learning Rounds")
    plt.xlabel("Round")
    plt.ylabel("Usage (%)")
    plt.legend()
    plt.grid(True)
    if pd.api.types.is_numeric_dtype(df["Round"]):
        plt.xticks(df["Round"].unique())
    plt.show()

## 14. Main execution

### 14.1 Explore client datasets

In [ ]:
for client_name, info in CLIENTS_INFO.items():
    print(f"Client: {client_name}")
    for split in ("train", "test", "val"):
        split_dir = os.path.join(info["path"], split)
        counts = count_images_by_class(split_dir, info["classes"])
        print(f"  {split.capitalize()} counts: {counts}")
        print(f"  Total {split} images: {sum(counts.values())}")
    print("-" * 20)

### 14.2 Run federated learning

In [ ]:
initial_model_path = f"{config.global_data_path}/9th11_Proposal_3_best_model.keras"

if os.path.exists(initial_model_path):
    global_model = load_fl_model(initial_model_path)
    print(f"Loaded initial global model from {initial_model_path}")
else:
    print(f"[WARN] Initial model not found at {initial_model_path}; building a fresh one.")
    global_model = model_fn((config.img_size, config.img_size, 3), len(config.all_classes), config.l2_lambda)
    global_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

global_model.summary()

global_model, global_history, client_history = run_federated_learning(
    global_model, CLIENTS_INFO, model_fn, config, use_delta=True,
)

final_model_path = os.path.join(config.model_dir, "FL_final_global_model.keras")
global_model.save(final_model_path)
print(f"Final global model saved to: {final_model_path}")

save_history(config, global_history, client_history)

### 14.3 Visualize training progress

In [ ]:
plot_training_curves(global_history, client_history)
plot_contribution_scores(config)
plot_gradient_similarity(config)
plot_resource_usage(os.path.join(config.resource_history_dir, "system_usage.csv"))

### 14.4 Evaluate the global model

In [ ]:
global_test_dir = os.path.join(config.global_data_path, "test")

# Evaluate a single round
single_round_report = evaluate_round_range(config, [config.rounds], global_test_dir, config.all_classes, title="FedPerGC")

# Evaluate a range of rounds (replaces the original notebook's copy-pasted per-round eval cells)
range_reports = evaluate_round_range(config, range(max(1, config.rounds - 8), config.rounds + 1), global_test_dir, config.all_classes, title="FedPerGC")

# Evaluate the final global model on each client's own test set
final_model = load_fl_model(final_model_path)
per_client_reports = evaluate_per_client(final_model, CLIENTS_INFO, config, title="FedPerGC alpha=0.5")

### 14.5 Benchmark inference performance

In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen.flow_from_directory(
    global_test_dir, target_size=(config.img_size, config.img_size), batch_size=config.batch_size,
    class_mode="categorical", classes=config.all_classes, shuffle=False,
)

benchmark_df = benchmark_model_with_format(final_model, test_generator, batch_sizes=(8, 16, 32, 64), runs=5, warmup_batches=2)
print(benchmark_df)

benchmark_output_path = os.path.join(config.resource_history_dir, f"{config.pass_name}_benchmark.csv")
os.makedirs(os.path.dirname(benchmark_output_path), exist_ok=True)
benchmark_df.to_csv(benchmark_output_path, index=False)
print(f"Benchmark results saved to: {benchmark_output_path}")